# My notes: full workflow — EDA to Tree-Based Models (Wine dataset)

Personal reference notebook. This time starting from the actual beginning: **look at the data first**, before touching any model code.

**The real shape of the whole project, in one line:**
`explore -> clean/prepare -> build -> train -> inspect -> predict -> evaluate -> tune (pre-pruning / post-pruning) -> ensemble (forest / boosting)`

Whenever I revisit this: the modeling part (Part 2 onward) is the "fun" part, but Part 1 (exploration) is where most real projects actually live or die. A great model on unexamined data is a trap - looks good in the notebook, falls apart in the real world.

## 0. Setup

In [ ]:
!pip install -q xgboost lightgbm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn import tree, ensemble, metrics
import xgboost as xgb
import lightgbm as lgb
import time

import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


# Part 1 — Explore and understand the data (EDA)

**Why I do this before any modeling:** I need to know if the data is trustworthy and what I'm actually working with. Specifically I'm checking for: missing values, class imbalance, weird outliers, redundant/correlated features, and whether anything looks like it might "leak" the answer.

**Note to self:** wine is a famous *clean* teaching dataset, so I won't find much wrong here — but I'm running the checks anyway, because on a REAL dataset I bring to a job, I wouldn't know that in advance. The habit matters more than this specific dataset.

## 1.1 Load the data and turn it into a normal table

**Reminder:** `load_wine()` returns a special sklearn "Bunch" object, not a plain table. I convert it to a pandas DataFrame immediately because pandas is much easier to explore with (`.head()`, `.describe()`, `.corr()`, etc.).

In [ ]:
wine = load_wine()

df = pd.DataFrame(wine.data, columns=wine.feature_names)
df['target'] = wine.target   # 0, 1, 2 -> the 3 grape varieties

feature_names = wine.feature_names
class_names = wine.target_names

df.head()


## 1.2 Shape and structure

**What I'm checking:** how many rows/columns do I actually have, and what type is each column? A classifier needs numeric features - if I saw an `object` dtype column here, that'd mean text data needing encoding before any tree could use it.

In [ ]:
print("Shape (rows, columns):", df.shape)
df.info()

# note: all 13 features + target are numeric (float64/int64) here - nothing to encode.
# on a real dataset I'd expect to see some 'object' dtype columns (categories, text) that need handling.


## 1.3 Missing values

**What I'm checking:** any column with missing data needs a decision - drop the rows, drop the column, or fill in ('impute') the gaps. Trees can technically handle some missingness depending on the library, but sklearn's DecisionTreeClassifier can NOT - it'll error out on NaNs, so this check matters here specifically.

In [ ]:
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values found.")

# if I DID find missing values here, my options would be:
#   df.dropna()                          - drop rows with any missing value
#   df['col'].fillna(df['col'].median())  - fill numeric gaps with the median (robust to outliers)
#   df['col'].fillna(df['col'].mode()[0]) - fill categorical gaps with the most common value


## 1.4 Class balance

**Why this matters a lot:** if one class was rare (say 95% class_0, 5% class_1), a model could get 95% accuracy just by always guessing class_0 - looking great while being useless. I need to know this BEFORE I trust any accuracy number later.

In [ ]:
counts = df['target'].value_counts().sort_index()
counts.index = class_names

print(counts)
counts.plot(kind='bar', title='Class balance', ylabel='count')
plt.show()

# wine is fairly balanced (59 / 71 / 48) - good, I don't need to worry about class imbalance techniques here


## 1.5 Distributions of each feature

**What I'm checking:** ranges, typical values, obvious outliers, and skew. Also - are features on wildly different scales?

**Note to self, important for later:** tree models don't care about feature scale (unlike e.g. linear regression or KNN) - a tree just asks "is this value above/below a threshold", so I do NOT need to standardize/normalize features before feeding them into a tree. Good to confirm that's true here.

In [ ]:
print(df.describe().T[['min', 'mean', 'max', 'std']])

df.drop(columns='target').hist(figsize=(14, 10), bins=20)
plt.tight_layout()
plt.show()

# proline ranges up into the hundreds/thousands, hue is under 2 - hugely different scales.
# confirms: no scaling needed for trees, but I'd need it if I were using a different model type.


## 1.6 Correlations

**What I'm checking two things:** (1) which features correlate strongly with the target - promising signal, but if one is suspiciously *too* perfectly correlated, that's a red flag for data leakage. (2) which features correlate strongly with EACH OTHER - redundant info, not necessarily a problem for trees, but useful context.

In [ ]:
corr_with_target = df.corr()['target'].drop('target').sort_values()
print(corr_with_target)

plt.figure(figsize=(10, 8))
plt.imshow(df.drop(columns='target').corr(), cmap='coolwarm', vmin=-1, vmax=1)
plt.xticks(range(len(feature_names)), feature_names, rotation=90)
plt.yticks(range(len(feature_names)), feature_names)
plt.colorbar(label='correlation')
plt.title('Feature correlation matrix')
plt.tight_layout()
plt.show()

# nothing is suspiciously close to +/-1.0 with target -> no obvious leakage.
# a few features correlate moderately with each other (e.g. flavanoids & total_phenols) - worth remembering,
# but trees handle correlated features fine (they just pick whichever one splits best at each node).


## 1.7 EDA takeaways (my summary before modeling)

- 178 rows, 13 numeric features, 3-class target - all numeric, no encoding needed
- No missing values - no imputation needed
- Classes are reasonably balanced (59/71/48) - plain accuracy is a fair metric here, no special imbalance handling needed
- Features are on very different scales - fine for trees, would matter for other model types
- No signs of data leakage in the correlation check

**Decision going into modeling:** this dataset needs almost no cleaning. I'm keeping all 13 features and moving straight to the train/test split. On a messier real dataset, this is where I'd add: dropping/imputing missing values, encoding categorical columns, removing duplicate rows, and maybe dropping any leaky columns I found.

# Part 2 — Build, train, and evaluate tree-based models

Now that I trust the data, this is the modeling pipeline I already know: build -> train -> inspect -> predict -> evaluate -> tune -> ensemble.

## 2.1 Train/test split

**Why:** never test on rows the model already saw during training. `train_test_split` holds back 20% as a fair, unseen "quiz." I'm reusing the same `df`/`wine` objects from the EDA above, so nothing needs reloading.

In [ ]:
X = df.drop(columns='target').values
y = df['target'].values

seed = 4000                # fixed seed for this whole notebook - keeps results reproducible and comparable
min_samples_leaf = 2       # don't create a leaf with fewer than this many samples
min_samples_split = 3      # don't try to split a node with fewer than this many samples

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=seed
)

print(f"Training rows: {X_train.shape[0]}, Test rows: {X_test.shape[0]}")


## 2.2 Build + train a baseline Decision Tree

**Key distinction to remember:** building (creating the object, no data touched) is not the same as training (`.fit()` - actually learns from data, mutates the tree in place).

In [ ]:
clf = tree.DecisionTreeClassifier(
    criterion='gini',       # how to judge "which question is best" at each split
    max_depth=None,         # no depth limit -> min_samples_leaf/split are my only overfitting guardrails
    min_samples_leaf=min_samples_leaf,
    min_samples_split=min_samples_split,
    random_state=seed
)

clf.fit(X_train, y_train)   # <- the actual learning step

print(clf)


## 2.3 Read the tree's learned rules

**Why:** sanity check - does the logic look reasonable? Read-only, doesn't change `clf`.

In [ ]:
print(tree.export_text(
    clf,
    feature_names=list(feature_names),
    class_names=[str(c) for c in class_names]
))


## 2.4 Predict + evaluate the baseline

**Reminder:** `.predict()` is read-only, unlike `.fit()`.

**How to read the report:** precision = "when I said class X, was I right?", recall = "of all real class X, how many did I catch?", f1 = blend of both.

In [ ]:
y_pred = clf.predict(X_test)

print("Baseline Decision Tree -- Classification Report\n")
print(metrics.classification_report(y_test, y_pred, target_names=class_names))

# <- my baseline number to beat


## 2.5 Pre-pruning via GridSearchCV

**Idea:** instead of guessing min_samples_leaf/split by hand, let the computer try every combination via cross-validation and pick the winner.

**Gotcha:** call `.fit()` on `grid_search`, NOT the raw tree.

In [ ]:
param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': list(range(2, 5)),
    'min_samples_leaf': list(range(2, 5)),
    'min_samples_split': list(range(2, 5)),
}

grid_search = GridSearchCV(estimator=tree.DecisionTreeClassifier(random_state=seed), param_grid=param_grid)
grid_search.fit(X_train, y_train)

best_params = grid_search.best_params_
print("Best pre-pruning parameters:", best_params)

prepruned_clf = tree.DecisionTreeClassifier(**best_params, random_state=seed)
prepruned_clf.fit(X_train, y_train)

y_pred = prepruned_clf.predict(X_test)
print("\nPre-Pruned Decision Tree -- Classification Report\n")
print(metrics.classification_report(y_test, y_pred, target_names=class_names))


## 2.6 Post-pruning, part 1: get the pruning path

**Different philosophy:** let the tree grow FULLY, then cut back. `ccp_alpha` = strictness dial, 0 = no pruning, higher = more aggressive.

In [ ]:
full_clf = tree.DecisionTreeClassifier(
    min_samples_leaf=min_samples_leaf,
    min_samples_split=min_samples_split,
    random_state=seed
)
path = full_clf.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas, impurities = path.ccp_alphas, path.impurities

plt.figure(figsize=(8, 6))
plt.plot(ccp_alphas, impurities, marker='o', drawstyle='steps-post')
plt.xlabel("ccp_alpha")
plt.ylabel("Total Leaf Impurity")
plt.title("Total Leaf Impurity vs ccp_alpha on Training Data")
plt.grid(True)
plt.show()


## 2.7 Post-pruning, part 2: train one tree per alpha

**Watch for:** training accuracy drops steadily as alpha rises. Test accuracy rises first, peaks, then collapses - the bias-variance tradeoff, visually.

In [ ]:
clfs, train_scores, test_scores = [], [], []

for ccp_alpha in ccp_alphas:
    c = tree.DecisionTreeClassifier(
        random_state=seed,
        min_samples_leaf=min_samples_leaf,
        min_samples_split=min_samples_split,
        ccp_alpha=ccp_alpha
    )
    c.fit(X_train, y_train)
    train_scores.append(c.score(X_train, y_train))
    test_scores.append(c.score(X_test, y_test))
    clfs.append(c)

plt.figure(figsize=(8, 6))
plt.plot(ccp_alphas, train_scores, marker='o', label='Train Accuracy', drawstyle='steps-post')
plt.plot(ccp_alphas, test_scores, marker='o', label='Test Accuracy', drawstyle='steps-post')
plt.xlabel('ccp_alpha')
plt.ylabel('Accuracy')
plt.title('Effect of ccp_alpha on Train and Test Accuracy')
plt.legend()
plt.grid(True)
plt.show()


## 2.8 Post-pruning, part 3: pick the winner

**3-step tiebreak rule:** (1) highest test accuracy, (2) smallest train/test gap, (3) biggest alpha. Tuple comparison + `max()` does all 3 steps in one line.

In [ ]:
def sort_key(i):
    gap = abs(train_scores[i] - test_scores[i])
    return (test_scores[i], -gap, ccp_alphas[i])

best_index = max(range(len(clfs)), key=sort_key)
best_alpha = ccp_alphas[best_index]
best_pruned_clf = clfs[best_index]

print(f"Best alpha: {best_alpha}")

y_pred = best_pruned_clf.predict(X_test)
print("\nPost-Pruned Decision Tree -- Classification Report\n")
print(metrics.classification_report(y_test, y_pred, target_names=class_names))


## 2.9 Random Forest

**Mental model:** many trees, each trained on a random sample of rows + random subset of features per split, voting together. Diversity across trees is what cancels out individual overfitting.

In [ ]:
rf = ensemble.RandomForestClassifier(n_estimators=20, random_state=seed)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print("Random Forest -- Classification Report\n")
print(metrics.classification_report(y_test, y_pred, target_names=class_names))


## 2.10 Feature importance

**What this tells me:** which columns the forest actually relied on - tracked automatically from impurity reduction across all trees. `np.argsort` gives the reorder-recipe, not sorted values, so I can keep feature_names lined up with scores.

In [ ]:
importances = rf.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(10, 6))
plt.barh(range(len(importances)), importances[indices])
plt.yticks(range(len(importances)), [feature_names[i] for i in indices])
plt.xlabel("Feature Importance")
plt.title("Random Forest - Feature Importances")
plt.tight_layout()
plt.show()


## 2.11 Boosting: AdaBoost / GradientBoosting / XGBoost / LightGBM

**Difference from Random Forest:** trees are sequential, not independent - each new tree fixes the previous ones' mistakes. Usually more accurate, usually slower (can't parallelize across trees the way bagging can).

In [ ]:
def build_boosting_model(name, n_estimators, random_state):
    """Factory function: given a name string, return the matching untrained boosting model."""
    if name == "adaboost":
        return ensemble.AdaBoostClassifier(n_estimators=n_estimators, random_state=random_state)
    elif name == "gradientboosting":
        return ensemble.GradientBoostingClassifier(n_estimators=n_estimators, random_state=random_state)
    elif name == "xgboost":
        return xgb.XGBClassifier(n_estimators=n_estimators, random_state=random_state)
    elif name == "lightgbm":
        return lgb.LGBMClassifier(n_estimators=n_estimators, random_state=random_state, verbose=-1)
    else:
        raise ValueError(f"Unknown model name '{name}'")

model_names = ["adaboost", "gradientboosting", "xgboost", "lightgbm"]
n_estimators = 20
n_runs = 50  # lowered from 500 for faster Colab runs - raise for more stable timing

accuracies = {}
avg_train_times = {}

for name in model_names:
    times = []
    for _ in range(n_runs):
        model = build_boosting_model(name, n_estimators, seed)
        start = time.time()
        model.fit(X_train, y_train)
        times.append(time.time() - start)
    avg_train_times[name] = sum(times) / n_runs

    model = build_boosting_model(name, n_estimators, seed)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracies[name] = metrics.accuracy_score(y_test, y_pred)

print("Accuracies:", accuracies)
print("Avg training times (s):", avg_train_times)


In [ ]:
colors = ['blue', 'orange', 'green', 'red']

plt.figure(figsize=(9, 5))
plt.bar(accuracies.keys(), accuracies.values(), color=colors)
plt.title("Accuracy Comparison of Boosting Classifiers on the Wine Dataset")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
for i, v in enumerate(accuracies.values()):
    plt.text(i, v + 0.01, f"{v:.2f}", ha='center', fontweight='bold')
plt.show()

plt.figure(figsize=(9, 5))
plt.bar(avg_train_times.keys(), avg_train_times.values(), color=colors)
plt.ylabel("Time (seconds)")
plt.title(f"Average Training Time ({n_runs} runs) for Boosting Classifiers")
for i, v in enumerate(avg_train_times.values()):
    plt.text(i, v + 0.001, f"{v:.3f}", ha='center', fontweight='bold')
plt.show()


## My final takeaways (fill in / update each time I revisit)

**Part 1 - EDA:** clean dataset, no missing values, balanced classes, no leakage - almost no cleaning needed here. On a real dataset this section would be longer (encoding, imputing, deduplicating).

**Part 2 - Modeling:**
- Baseline tree: ~0.75 accuracy -> a single unlimited-depth tree overfits
- Pre-pruned tree: grid search found better stopping rules than hand-picked guesses
- Post-pruned tree: same idea, different method (grow full then cut back via ccp_alpha)
- Random Forest: biggest jump, ~0.94 -> voting across many diverse trees beats any single tuned tree
- Boosting: compare accuracy vs training time, pick based on what the project actually needs

**General lesson I keep relearning:** exploration comes first, always. A model is only as trustworthy as the data it was checked against before training even started. And once modeling starts: begin simple, diagnose the specific problem (usually overfitting), THEN reach for pruning or ensembling to fix that specific problem - don't jump straight to the fanciest model.